In [1]:
# ============================================================
# WEEK 4 — Policy Evaluation & Business Dashboard
# Project 2: Travel & Hospitality — RL Dynamic Pricing
# Intern Branch: preeti-dev | Infotact Solutions
#
# Roadmap requirement:
#   "Run 1,000 simulated booking seasons to evaluate the DQN
#    agent against the naive baselines. Plot the 'Price
#    Trajectory' over time, proving that the agent learned
#    complex behaviors (like dropping prices near the deadline
#    to clear remaining stock)."
# ============================================================
 
 
# ── CELL 1: Install & Import Libraries ───────────────────
import subprocess, sys
 
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
 
for lib in ["gymnasium", "torch", "numpy", "matplotlib", "seaborn", "pandas", "tqdm"]:
    install(lib)
 
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import pickle
import os
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')
 
os.makedirs('../reports', exist_ok=True)
os.makedirs('../data',    exist_ok=True)
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ All libraries imported successfully")
 

✅ All libraries imported successfully


In [2]:
# ── CELL 2: Re-define Environment (standalone copy) ──────
MARKET_EVENTS = {
    0: {'name': 'Normal',          'demand_multiplier': 1.0,  'color': '#60a5fa'},
    1: {'name': 'Holiday Surge',   'demand_multiplier': 1.6,  'color': '#34d399'},
    2: {'name': 'Competitor Sale', 'demand_multiplier': 0.6,  'color': '#f87171'},
    3: {'name': 'Bad Weather',     'demand_multiplier': 0.75, 'color': '#fbbf24'},
}
EVENT_PROBS = [0.65, 0.12, 0.15, 0.08]
 
CUSTOMER_SEGMENTS = {
    'business':     {'price_sensitivity':0.3, 'booking_window':'any',   'base_probability':0.25},
    'leisure':       {'price_sensitivity':0.9, 'booking_window':'early', 'base_probability':0.50},
    'last_minute':   {'price_sensitivity':0.2, 'booking_window':'late',  'base_probability':0.25},
}
 
SAFETY_CONFIG = {
    'eco_floor_idx':1, 'eco_ceiling_idx':7,
    'biz_floor_idx':1, 'biz_ceiling_idx':7,
    'max_daily_swing':3, 'violation_penalty':25.0
}
 
class ContextualAirlinePricingEnv(gym.Env):
    metadata = {'render_modes': ['human']}
    ECO_PRICES        = [50, 100, 150, 200, 250, 300, 350, 400]
    BIZ_PRICES        = [300, 450, 600, 750, 900, 1050, 1100, 1200]
    COMPETITOR_PRICES = [80, 120, 160, 200, 240, 280, 320, 360]
 
    def __init__(self, eco_seats=40, biz_seats=10, total_days=30,
                 safety_config=SAFETY_CONFIG, render_mode=None):
        super().__init__()
        self.eco_seats_init, self.biz_seats_init = eco_seats, biz_seats
        self.total_days, self.render_mode, self.safety = total_days, render_mode, safety_config
        self.action_space = spaces.MultiDiscrete([len(self.ECO_PRICES), len(self.BIZ_PRICES)])
        self.observation_space = spaces.Box(
            low=np.array([0,0,0,0,0], dtype=np.float32),
            high=np.array([eco_seats,biz_seats,total_days,
                          len(self.COMPETITOR_PRICES)-1,len(MARKET_EVENTS)-1], dtype=np.float32),
            dtype=np.float32)
        self._init_state()
 
    def _init_state(self):
        self.eco_seats, self.biz_seats = self.eco_seats_init, self.biz_seats_init
        self.days_left = self.total_days
        self.total_revenue = self.eco_revenue = self.biz_revenue = 0.0
        self.market_event_id, self.competitor_idx = 0, 3
        self.prev_eco_idx = self.prev_biz_idx = 4
        self.safety_violations = 0
        self.history = []
 
    def _sample_market_event(self):
        return int(np.random.choice(len(MARKET_EVENTS), p=EVENT_PROBS))
 
    def _update_competitor_price(self):
        drift = np.random.choice([-1,0,0,1])
        self.competitor_idx = int(np.clip(self.competitor_idx+drift, 0, len(self.COMPETITOR_PRICES)-1))
 
    def _apply_safety_bounds(self, eco_idx, biz_idx):
        violated = False
        safe_eco = int(np.clip(eco_idx, self.safety['eco_floor_idx'], self.safety['eco_ceiling_idx']))
        safe_biz = int(np.clip(biz_idx, self.safety['biz_floor_idx'], self.safety['biz_ceiling_idx']))
        if safe_eco != eco_idx or safe_biz != biz_idx: violated = True
        max_swing = self.safety['max_daily_swing']
        if abs(safe_eco - self.prev_eco_idx) > max_swing:
            safe_eco = int(np.clip(self.prev_eco_idx + max_swing*np.sign(safe_eco-self.prev_eco_idx),
                                   0, len(self.ECO_PRICES)-1)); violated = True
        if abs(safe_biz - self.prev_biz_idx) > max_swing:
            safe_biz = int(np.clip(self.prev_biz_idx + max_swing*np.sign(safe_biz-self.prev_biz_idx),
                                   0, len(self.BIZ_PRICES)-1)); violated = True
        return safe_eco, safe_biz, violated
 
    def _segment_demand(self, price, days_left, seat_class, event_mult):
        max_price = max(self.BIZ_PRICES) if seat_class=='biz' else max(self.ECO_PRICES)
        total = 0
        for seg_name, seg in CUSTOMER_SEGMENTS.items():
            if seg['booking_window']=='late' and days_left>7: continue
            if seg['booking_window']=='early' and days_left<3: continue
            price_factor = max(0.0, 1.0 - seg['price_sensitivity']*(price/max_price))
            if seg_name=='last_minute': urgency = np.exp(-days_left/3)
            elif seg_name=='business':  urgency = 0.6+0.4*np.exp(-days_left/15)
            else:                      urgency = 1.0-0.6*np.exp(-days_left/20)
            competitor_eco = self.COMPETITOR_PRICES[self.competitor_idx]
            comp_factor = float(np.clip(1.0+0.3*(competitor_eco-price)/max_price, 0.5, 1.8))
            prob = seg['base_probability']*price_factor*urgency*comp_factor*event_mult + np.random.uniform(-0.05,0.05)
            prob = float(np.clip(prob, 0.0, 1.0))
            arrivals = np.random.randint(0,4)
            total += sum(np.random.random()<prob for _ in range(arrivals))
        return total
 
    def _get_obs(self):
        return np.array([self.eco_seats,self.biz_seats,self.days_left,
                         self.competitor_idx,self.market_event_id], dtype=np.float32)
 
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._init_state()
        self.market_event_id = self._sample_market_event()
        self._update_competitor_price()
        return self._get_obs(), {}
 
    def step(self, action):
        raw_eco, raw_biz = int(action[0]), int(action[1])
        eco_idx, biz_idx, violated = self._apply_safety_bounds(raw_eco, raw_biz)
        eco_price, biz_price = self.ECO_PRICES[eco_idx], self.BIZ_PRICES[biz_idx]
        event = MARKET_EVENTS[self.market_event_id]; event_mult = event['demand_multiplier']
        eco_bk = min(self._segment_demand(eco_price,self.days_left,'eco',event_mult), self.eco_seats)
        biz_bk = min(self._segment_demand(biz_price,self.days_left,'biz',event_mult), self.biz_seats)
        eco_rev, biz_rev = eco_price*eco_bk, biz_price*biz_bk
        daily_rev = eco_rev+biz_rev
        penalty = self.safety['violation_penalty'] if violated else 0.0
        if violated: self.safety_violations += 1
        reward = daily_rev - penalty
        self.eco_seats -= eco_bk; self.biz_seats -= biz_bk
        self.total_revenue += daily_rev; self.eco_revenue += eco_rev; self.biz_revenue += biz_rev
        self.days_left -= 1; self.prev_eco_idx, self.prev_biz_idx = eco_idx, biz_idx
        self.market_event_id = self._sample_market_event(); self._update_competitor_price()
        self.history.append({
            'day':self.total_days-self.days_left, 'days_left':self.days_left+1,
            'eco_price':eco_price,'biz_price':biz_price,
            'competitor_price':self.COMPETITOR_PRICES[self.competitor_idx],
            'market_event':event['name'],'eco_bookings':eco_bk,'biz_bookings':biz_bk,
            'eco_rev':eco_rev,'biz_rev':biz_rev,'daily_revenue':daily_rev,
            'safety_violated':violated,'eco_seats_left':self.eco_seats,
            'biz_seats_left':self.biz_seats,'total_revenue':self.total_revenue})
        terminated = (self.days_left==0 or (self.eco_seats==0 and self.biz_seats==0))
        return self._get_obs(), reward, terminated, False, {}
 
    def get_history_df(self):
        return pd.DataFrame(self.history)
 
print("✅ Environment re-loaded (standalone copy)")

✅ Environment re-loaded (standalone copy)


In [3]:
# ── CELL 3: Re-define DQN Architecture & Load Trained Weights ──
N_ECO_ACTIONS = len(ContextualAirlinePricingEnv.ECO_PRICES)
N_BIZ_ACTIONS = len(ContextualAirlinePricingEnv.BIZ_PRICES)
N_ACTIONS     = N_ECO_ACTIONS * N_BIZ_ACTIONS
 
def action_to_multi(action_idx):
    return action_idx // N_BIZ_ACTIONS, action_idx % N_BIZ_ACTIONS
 
class QNetwork(nn.Module):
    def __init__(self, state_dim=5, n_actions=N_ACTIONS, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, n_actions))
    def forward(self, x):
        return self.net(x)
 
state_scale = np.array([40, 10, 30, 7, 3], dtype=np.float32)
 
q_net = QNetwork().to(device)
q_net.load_state_dict(torch.load('../models/dqn_weights.pth', map_location=device))
q_net.eval()
 
def dqn_choose_action(obs):
    state_norm = obs / state_scale
    state_t = torch.FloatTensor(state_norm).unsqueeze(0).to(device)
    with torch.no_grad():
        q_values = q_net(state_t)
    return int(torch.argmax(q_values).item())
 
print("✅ Trained DQN weights loaded from models/dqn_weights.pth")

✅ Trained DQN weights loaded from models/dqn_weights.pth


In [4]:
# ── CELL 4: Re-define Baseline Agents ────────────────────
def fixed_price_agent(env, eco_idx=4, biz_idx=4):
    obs, _ = env.reset()
    done = False
    while not done:
        obs, reward, done, _, _ = env.step(np.array([eco_idx, biz_idx]))
    return env
 
def time_based_agent(env):
    obs, _ = env.reset()
    done = False
    while not done:
        days_left = obs[2]
        progress = 1 - (days_left / env.total_days)
        eco_idx = int(np.clip(7 - progress*6, 1, 7))
        biz_idx = int(np.clip(7 - progress*6, 1, 7))
        obs, reward, done, _, _ = env.step(np.array([eco_idx, biz_idx]))
    return env
 
def emsr_b_agent(env):
    obs, _ = env.reset()
    done = False
    expected_total_eco_demand, expected_total_biz_demand = 35, 8
    while not done:
        eco_seats, biz_seats, days_left = obs[0], obs[1], obs[2]
        expected_remaining_eco = expected_total_eco_demand * (days_left / env.total_days)
        expected_remaining_biz = expected_total_biz_demand * (days_left / env.total_days)
        eco_ratio = eco_seats / max(expected_remaining_eco, 1)
        biz_ratio = biz_seats / max(expected_remaining_biz, 1)
        eco_idx = int(np.clip(round(7 - eco_ratio*3), 1, 7))
        biz_idx = int(np.clip(round(7 - biz_ratio*3), 1, 7))
        obs, reward, done, _, _ = env.step(np.array([eco_idx, biz_idx]))
    return env
 
def dqn_agent_runner(env):
    obs, _ = env.reset()
    done = False
    while not done:
        action_idx = dqn_choose_action(obs)
        eco_idx, biz_idx = action_to_multi(action_idx)
        obs, reward, done, _, _ = env.step(np.array([eco_idx, biz_idx]))
    return env
 
print("✅ All agent runners defined (Fixed, Time-Based, EMSR-b, DQN)")
 

✅ All agent runners defined (Fixed, Time-Based, EMSR-b, DQN)


In [5]:
# ── CELL 5: Run 1,000 Simulated Booking Seasons ──────────
# ★ This is the core Week 4 requirement — a large-scale,
# statistically robust evaluation across all strategies.
 
N_SEASONS = 1000
 
strategies = {
    'Fixed Price'  : fixed_price_agent,
    'Time-Based'   : time_based_agent,
    'EMSR-b'       : emsr_b_agent,
    'DQN'          : dqn_agent_runner,
}
 
print(f"Running {N_SEASONS} simulated booking seasons per strategy...")
print("This is a large evaluation — will take a few minutes.")
print("=" * 60)
 
results = {name: [] for name in strategies}
safety_results = {name: [] for name in strategies}
sell_through_results = {name: [] for name in strategies}
 
env_sim = ContextualAirlinePricingEnv()
 
for name, agent_fn in strategies.items():
    revs, safety, sell_through = [], [], []
    for season in tqdm(range(N_SEASONS), desc=f"{name:<14}", leave=False):
        np.random.seed(season + 20000)
        env_result = agent_fn(env_sim)
        revs.append(env_result.total_revenue)
        safety.append(env_result.safety_violations)
        total_seats = env_result.eco_seats_init + env_result.biz_seats_init
        sold = total_seats - (env_result.eco_seats + env_result.biz_seats)
        sell_through.append(sold / total_seats * 100)
 
    results[name] = revs
    safety_results[name] = safety
    sell_through_results[name] = sell_through
    print(f"  ✅ {name:<14} — Mean Revenue: ${np.mean(revs):>10,.0f}  |  "
          f"Sell-through: {np.mean(sell_through):.1f}%")
 
print("=" * 60)
print(f"✅ {N_SEASONS}-season evaluation complete for all 4 strategies")

Running 1000 simulated booking seasons per strategy...
This is a large evaluation — will take a few minutes.


  ✅ Fixed Price    — Mean Revenue: $    11,353  |  Sell-through: 45.1%


  ✅ Time-Based     — Mean Revenue: $    10,196  |  Sell-through: 46.3%


  ✅ EMSR-b         — Mean Revenue: $    10,127  |  Sell-through: 57.3%


  ✅ DQN            — Mean Revenue: $    11,304  |  Sell-through: 41.8%
✅ 1000-season evaluation complete for all 4 strategies


In [6]:
# ── CELL 6: Statistical Summary Table ────────────────────
summary_rows = []
for name in strategies:
    revs = results[name]
    summary_rows.append({
        'Strategy'        : name,
        'Mean Revenue'    : round(np.mean(revs), 0),
        'Median Revenue'  : round(np.median(revs), 0),
        'Std Revenue'     : round(np.std(revs), 0),
        'Min Revenue'     : round(np.min(revs), 0),
        'Max Revenue'     : round(np.max(revs), 0),
        'Sell-Through %'  : round(np.mean(sell_through_results[name]), 1),
        'Safety Blocks/Season': round(np.mean(safety_results[name]), 2)
    })
 
summary_df = pd.DataFrame(summary_rows).sort_values('Mean Revenue', ascending=False).reset_index(drop=True)
 
dqn_rev = summary_df.loc[summary_df['Strategy']=='DQN','Mean Revenue'].values[0]
summary_df['vs DQN (%)'] = ((summary_df['Mean Revenue'] - dqn_rev) / dqn_rev * 100).round(2)
 
print("=" * 90)
print(f"   1,000-SEASON EVALUATION — STATISTICAL SUMMARY")
print("=" * 90)
print(summary_df.to_string(index=False))
print("=" * 90)

   1,000-SEASON EVALUATION — STATISTICAL SUMMARY
   Strategy  Mean Revenue  Median Revenue  Std Revenue  Min Revenue  Max Revenue  Sell-Through %  Safety Blocks/Season  vs DQN (%)
Fixed Price       11353.0         11750.0       1886.0       4800.0      15750.0            45.1                  0.00        0.43
        DQN       11304.0         11450.0       2731.0       3350.0      16750.0            41.8                  0.00        0.00
 Time-Based       10196.0         10450.0       1887.0       3250.0      15750.0            46.3                  0.00       -9.80
     EMSR-b       10127.0         10400.0       1677.0       4200.0      15200.0            57.3                  0.11      -10.41


In [ ]:
# ── CELL 7: Revenue Distribution — All Strategies ────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.patch.set_facecolor('#0f1a2e')
for ax in axes:
    ax.set_facecolor('#0f1a2e')
    for spine in ax.spines.values(): spine.set_edgecolor('#1e3254')
    ax.tick_params(colors='#94a3b8')
    ax.xaxis.label.set_color('#64748b'); ax.yaxis.label.set_color('#64748b')
    ax.title.set_color('#e2e8f0')
 
strat_colors = {'Fixed Price':'#94a3b8','Time-Based':'#fbbf24','EMSR-b':'#f87171','DQN':'#34d399'}
 
# Boxplot
box_data = [results[n] for n in strategies]
bp = axes[0].boxplot(box_data, labels=list(strategies.keys()), patch_artist=True,
                     medianprops=dict(color='#0b0f1a', linewidth=2))
for patch, name in zip(bp['boxes'], strategies):
    patch.set_facecolor(strat_colors[name]); patch.set_alpha(0.75)
axes[0].set_ylabel('Total Revenue per Season ($)')
axes[0].set_title(f'Revenue Distribution — {N_SEASONS} Simulated Seasons')
axes[0].set_xticklabels(list(strategies.keys()), rotation=12, ha='right', fontsize=9)
axes[0].grid(alpha=0.2, color='#1e3254', axis='y')
 
# Bar with error bars
names  = list(strategies.keys())
means  = [np.mean(results[n]) for n in names]
stds   = [np.std(results[n])  for n in names]
colors_b = [strat_colors[n] for n in names]
bars = axes[1].bar(names, means, yerr=stds, capsize=6, color=colors_b,
                   alpha=0.85, edgecolor='#0f1a2e', width=0.6)
for bar, val in zip(bars, means):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+150,
                f'${val:,.0f}', ha='center', fontweight='bold',
                fontsize=10, color='#e2e8f0')
axes[1].set_ylabel('Mean Revenue ± Std ($)')
axes[1].set_title(f'Mean Revenue Comparison — {N_SEASONS} Seasons')
axes[1].set_xticklabels(names, rotation=12, ha='right', fontsize=9)
axes[1].grid(alpha=0.2, color='#1e3254', axis='y')
 
plt.suptitle('★ Large-Scale Policy Evaluation (1,000 Booking Seasons)',
             fontsize=14, fontweight='bold', color='#e2e8f0')
plt.tight_layout()
plt.savefig('../reports/1000_season_evaluation.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1a2e')
plt.show()
print("✅ Saved → reports/1000_season_evaluation.png")